# Normal-RAG2Graph 後端 Walkthrough（一步一步跑）

這份 notebook 把 `main` 分支上的後端拆成 9 個可獨立執行的步驟，方便逐 cell 觀察 RAG 與 GraphRAG 的內部資料流。

> 本 notebook **不啟動 FastAPI**，只示範核心邏輯（Parsing → Chunking → Embedding → 向量檢索 → LLM 生成 → KG 抽取 → SQLite 寫入 → 圖譜查詢 → Hybrid 高亮）。

## 學習目標
1. 看清楚一份文字怎麼變成向量、又怎麼被檢索回來
2. 看到 LLM 在 RAG 中扮演的角色（embedding、生成、結構化抽取）
3. 理解知識圖譜如何從同一份 chunk 衍生
4. 為 Stage 2 GraphRAG 教學打底

## 對應的後端原始碼
- `backend/app/rag_engine.py`（核心邏輯）
- `backend/app/graph_db.py`（SQLite 圖庫）
- `backend/app/main.py`（FastAPI 包裝；本 notebook 不涵蓋）


## Step 0 — 環境準備

依專案規範，後端套件**一律走 uv** 管理（不要用 pip）。下面這個 cell 會把 notebook 需要的依賴透過 `uv add` 補進 `backend/pyproject.toml`。

如果你已經跑過一次，重複跑也不會壞事；uv 會跳過已存在的依賴。


In [2]:
# 透過 uv 為 backend 補齊依賴（會更新 backend/pyproject.toml 與 uv.lock）
import subprocess, sys, pathlib

BACKEND_DIR = pathlib.Path("backend").resolve()
PACKAGES = [
    "chromadb>=1.5.7",
    "google-generativeai>=0.8.6",
    "langchain-text-splitters>=1.1.1",
    "pypdf>=6.10.0",
]

result = subprocess.run(
    ["uv", "add", *PACKAGES],
    cwd=BACKEND_DIR,
    capture_output=True, text=True,
)
print("STDOUT:", result.stdout[-2000:])
print("STDERR:", result.stderr[-1000:])
print("returncode =", result.returncode)


STDOUT: 
STDERR: warning: `VIRTUAL_ENV=/Users/kevinluo/google-agent-ecosystem/Antigravity-work/Normal-RAG2Graph-Project/.venv` does not match the project environment path `.venv` and will be ignored; use `--active` to target the active environment instead
Resolved 117 packages in 0.56ms
Audited 115 packages in 0.06ms

returncode = 0


為了讓 notebook（在專案根目錄）也能 import 這些套件，下一個 cell 會把 `backend/.venv` 的 site-packages 注入到目前 Python 的 `sys.path`。

> 之所以這樣處理，是因為 notebook 用的 kernel 不一定就是 `backend/.venv`。如果你習慣直接在 backend venv 裡開 Jupyter，可以略過這個 cell。


In [3]:
import sys, sysconfig, pathlib

BACKEND_VENV = pathlib.Path("backend/.venv").resolve()
# 找到 venv 內對應 Python 版本的 site-packages
candidates = list(BACKEND_VENV.glob("lib/python*/site-packages"))
if not candidates:
    raise RuntimeError(f"找不到 backend venv 的 site-packages：{BACKEND_VENV}")
site_pkgs = str(candidates[0])
if site_pkgs not in sys.path:
    sys.path.insert(0, site_pkgs)
print("已注入：", site_pkgs)


已注入： /Users/kevinluo/google-agent-ecosystem/Antigravity-work/Normal-RAG2Graph-Project/backend/.venv/lib/python3.13/site-packages


### 載入 GEMINI_API_KEY

從專案根目錄的 `.env` 讀取，避免額外引入 `python-dotenv`。

> 安全提醒：`.env` 已在 `.gitignore`，請勿把實際 key 貼到 cell output 或 commit 出去。


In [4]:
import os, pathlib

env_path = pathlib.Path(".env")
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

api_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
assert api_key, "請在專案根的 .env 設定 GEMINI_API_KEY 或 GOOGLE_API_KEY"
print("API key 已載入，長度 =", len(api_key))


API key 已載入，長度 = 39


## Step 1 — 文件解析（Parsing）

實務上後端會接收 `.pdf` 或 `.txt`：
- PDF → `pypdf.PdfReader` 逐頁 `extract_text()`
- TXT → 直接 `open(...).read()`

這裡為了 demo 可重現，**直接內建一段示範文字**。內容刻意挑選含有公司、人物、地點、年份等實體，後面 Step 6 KG 抽取才會有東西可看。


In [5]:
SAMPLE_TEXT = """
Anthropic 是一家位於美國舊金山的人工智慧公司，由 Dario Amodei 與 Daniela Amodei
於 2021 年共同創立。Anthropic 開發了 Claude 系列大型語言模型，包含 Claude 3、
Claude 3.5 Sonnet，以及最新的 Claude 4 系列。Anthropic 的研究方向以 AI 安全
（AI Safety）與可解釋性（Interpretability）為核心。

Google DeepMind 由 Demis Hassabis 領導，於 2023 年由 Google Brain 與 DeepMind
合併而成，總部位於英國倫敦。DeepMind 推出的 Gemini 模型系列是 Anthropic Claude
的主要競爭對手之一，並廣泛應用在 Google Search、Workspace 與 Android 產品線中。

OpenAI 由 Sam Altman 擔任執行長，總部位於美國舊金山，於 2015 年創立。其代表產品
是 ChatGPT 以及 GPT 系列模型。OpenAI 與 Microsoft 之間有深度合作關係，
Microsoft 透過 Azure OpenAI Service 將 GPT 模型整合進 Bing 與 Copilot 產品線。
"""

# 模擬一份檔案
DOC_ID = "demo_doc_001"
FILE_NAME = "ai_companies.txt"

print(f"模擬文件 {FILE_NAME}（doc_id={DOC_ID}）載入完成，總字數 = {len(SAMPLE_TEXT)}")


模擬文件 ai_companies.txt（doc_id=demo_doc_001）載入完成，總字數 = 572


## Step 2 — Chunking

`RecursiveCharacterTextSplitter` 會用一組分隔符（`\n\n` → `\n` → 空白 → 字）逐層切，
盡量在語意邊界斷開。`chunk_size=1000`、`chunk_overlap=200` 是 main 後端用的設定。

> 這裡示範文字偏短，會切成 1～2 個 chunks，但程式邏輯跟正式檔案完全一致。


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
chunks = text_splitter.split_text(SAMPLE_TEXT)

print(f"切出 {len(chunks)} 個 chunks\n")
for i, c in enumerate(chunks):
    print(f"--- chunk {i} （{len(c)} 字）---")
    print(c[:200] + ("..." if len(c) > 200 else ""))
    print()


ImportError: module ''langchain_core._api'.'deprecation'' not found (No module named 'pydantic_core._pydantic_core')

## Step 3 — Embedding 與寫入 ChromaDB

要點：
- `chromadb.PersistentClient(path="./local_chromadb")` → 跟後端共用同一份持久化資料夾
- Embedding 模型強制使用 `models/gemini-embedding-001`（`.agents/rules.md` 規定）
- 每個 chunk 帶 metadata（`doc_id`、`file_name`、`chunk_index`）方便之後反查／刪除

> ⚠️ 為什麼**不用** `chromadb.utils.embedding_functions.GoogleGenerativeAiEmbeddingFunction`？
>
> 截至本 notebook 撰寫時，`chromadb >=1.5.x` 內建的 wrapper 會傳 `client_options={"headers": ...}` 給 `genai.configure()`，但官方 `google-generativeai` 0.8.x 不接受這個欄位（且整個 `google-generativeai` 套件已 deprecated，官方改名成 `google-genai`）。為了避免被版本對齊綁住，這邊**自己寫一個 `EmbeddingFunction`**，直接呼叫 `genai.embed_content`。教學上也更透明。


In [ ]:
import chromadb
from chromadb.api.types import Documents, Embeddings, EmbeddingFunction
import google.generativeai as genai

genai.configure(api_key=api_key)

class GeminiEmbeddingFunction(EmbeddingFunction[Documents]):
    """直接呼叫 genai.embed_content，避開 Chroma 內建 wrapper 的版本相容問題。"""

    def __init__(self, model_name: str = "models/gemini-embedding-001",
                 task_type: str = "RETRIEVAL_DOCUMENT"):
        self.model_name = model_name
        self.task_type = task_type

    def __call__(self, input: Documents) -> Embeddings:
        result = genai.embed_content(
            model=self.model_name,
            content=list(input),
            task_type=self.task_type,
        )
        emb = result["embedding"]
        # 單筆時 API 會回傳 list[float]；多筆時回傳 list[list[float]]，這裡統一成後者
        if emb and isinstance(emb[0], (int, float)):
            emb = [emb]
        return emb


chroma_client = chromadb.PersistentClient(path="./local_chromadb")
embedding_fn = GeminiEmbeddingFunction()

collection = chroma_client.get_or_create_collection(
    name="documents_collection",
    embedding_function=embedding_fn,
)

# 為了 demo 可重複執行，先把上一輪的 demo 文件清掉
existing = collection.get(where={"doc_id": DOC_ID})
if existing["ids"]:
    collection.delete(ids=existing["ids"])
    print(f"清除上一輪殘留 chunks：{len(existing['ids'])} 筆")

ids = [f"{DOC_ID}_chunk_{i}" for i in range(len(chunks))]
metadatas = [
    {"doc_id": DOC_ID, "file_name": FILE_NAME, "chunk_index": i}
    for i in range(len(chunks))
]

collection.add(documents=chunks, metadatas=metadatas, ids=ids)
print(f"寫入完成：{len(ids)} 個 chunks，集合內現存 {collection.count()} 筆")


## Step 4 — 向量檢索（Retrieval）

對使用者問題做 embedding，再從 ChromaDB 撈最近的 top-k chunks。

`distances` 是內部距離（Chroma 預設 L2/cosine 之一）；後端用 `1.0 - distance` 當「相似分數」做 UI 顯示，
這只是粗略示意，**不是嚴格的 similarity score**，做研究時要小心。


In [ ]:
USER_QUERY = "Anthropic 是誰創立的？跟哪間公司是競爭對手？"

results = collection.query(
    query_texts=[USER_QUERY],
    n_results=min(5, collection.count()),
    include=["documents", "metadatas", "distances"],
)

retrieved_chunks = []
context_text = ""
for idx, doc in enumerate(results["documents"][0]):
    meta = results["metadatas"][0][idx]
    distance = results["distances"][0][idx]
    chunk_id = results["ids"][0][idx]
    retrieved_chunks.append({
        "id": chunk_id,
        "source": meta.get("file_name", "Unknown"),
        "score": 1.0 - distance,
        "text_snippet": doc[:200] + ("..." if len(doc) > 200 else ""),
    })
    context_text += f"\n\n--- 來源: {meta.get('file_name')} ---\n{doc}"

for c in retrieved_chunks:
    print(f"[{c['id']}] score={c['score']:.4f}  source={c['source']}")
    print(c["text_snippet"])
    print()


## Step 5 — LLM 生成（Naive RAG）

把上一步檢索到的 `context_text` 拼進 system prompt，然後丟給 `gemini-2.5-flash` 生成回答。

這個就是「普通 RAG」最關鍵的一步：**檢索結果作為上下文，模型只能根據上下文作答**。
若上下文沒有資訊就請模型承認不知道，避免幻覺。


In [ ]:
import google.generativeai as genai

genai.configure(api_key=api_key)
gen_model = genai.GenerativeModel("gemini-2.5-flash")

system_prompt = f"""你是一個精準且專業的問答助手。請根據以下檢索到的「參考上下文」來回答使用者的問題。
如果參考上下文中沒有足夠的資訊，請誠實地回答您不知道，不要編造資訊。
無論使用者的問題語言為何，請一律使用繁體中文（zh-TW）來回答。

【參考上下文開始】
{context_text}
【參考上下文結束】
"""
prompt = system_prompt + f"\n使用者問題: {USER_QUERY}\n回答: "

response = gen_model.generate_content(prompt)
print("=== Gemini 回答 ===")
print(response.text)


## Step 6 — 知識圖譜抽取（KG Extraction）

對每個 chunk 用 Gemini 在 `response_mime_type="application/json"` 模式下
抽出 `(source, target, relationship)` 三元組。

> 後端為了避免 demo 等太久 / 觸發 API rate limit，**只對前 3 個 chunks** 抽 KG。這裡保留同樣行為。


In [ ]:
import json

kg_model = genai.GenerativeModel(
    "gemini-2.5-flash",
    generation_config={"response_mime_type": "application/json"},
)

def extract_relations(chunk_text: str) -> list[dict]:
    prompt = f"""
請從以下文本中提取出核心實體 (Entity) 以及實體間的關係 (Relationship)。
將結果封裝為 JSON 格式，如下所示：
{{
    "relations": [
        {{"source": "實體A", "target": "實體B", "relationship": "關係描述"}}
    ]
}}
文本：
{chunk_text}
"""
    res = kg_model.generate_content(prompt)
    data = json.loads(res.text)
    return data.get("relations", [])

# 為了示範完整，這裡對所有 chunks 都跑（demo 文字才 1～2 chunks，不會慢）
all_relations = []
for i, chunk in enumerate(chunks[:3]):
    chunk_id = f"{DOC_ID}_chunk_{i}"
    rels = extract_relations(chunk)
    print(f"chunk {i} → 抽出 {len(rels)} 條關係")
    for r in rels:
        print(f"  ({r.get('source')}) -[{r.get('relationship')}]-> ({r.get('target')})")
        all_relations.append({**r, "chunk_id": chunk_id})

print(f"\n總計 {len(all_relations)} 條關係")


## Step 7 — 寫入 SQLite GraphDB

main 後端用 `local_graph.db` 存圖譜，兩張表：

- `entities (id, name, type)` — 節點
- `relations (id, source_name, target_name, relation_type, chunk_id)` — 邊（含「這條邊是哪個 chunk 抽出來的」）

`chunk_id` 這欄是 GraphRAG 反查高亮的關鍵：點圖譜上的節點時可以反查回原文片段。


In [ ]:
import sqlite3, pathlib

DB_PATH = "./local_graph.db"
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS entities (
    id   TEXT PRIMARY KEY,
    name TEXT UNIQUE,
    type TEXT
)
""")
cur.execute("""
CREATE TABLE IF NOT EXISTS relations (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    source_name   TEXT,
    target_name   TEXT,
    relation_type TEXT,
    chunk_id      TEXT
)
""")

# demo 重跑時先清掉這份文件相關的 relations
cur.execute("DELETE FROM relations WHERE chunk_id LIKE ?", (f"{DOC_ID}_chunk_%",))

for r in all_relations:
    src, tgt = r.get("source"), r.get("target")
    if not src or not tgt:
        continue
    cur.execute("INSERT OR IGNORE INTO entities (id, name, type) VALUES (?, ?, 'Unknown')", (src, src))
    cur.execute("INSERT OR IGNORE INTO entities (id, name, type) VALUES (?, ?, 'Unknown')", (tgt, tgt))
    cur.execute(
        "INSERT INTO relations (source_name, target_name, relation_type, chunk_id) VALUES (?, ?, ?, ?)",
        (src, tgt, r.get("relationship", "associated_with"), r["chunk_id"]),
    )

conn.commit()

cur.execute("SELECT COUNT(*) FROM entities")
n_ent = cur.fetchone()[0]
cur.execute("SELECT COUNT(*) FROM relations")
n_rel = cur.fetchone()[0]
print(f"SQLite 寫入完成 — entities={n_ent}, relations={n_rel}")
conn.close()


## Step 8 — 圖譜查詢

模擬後端 `GET /api/v1/graph` 的回傳格式（`{nodes: [...], links: [...]}`，前端 force-graph 可直接吃）。
最後再示範「點某個節點 → 反查它出現過的 chunk_id」的邏輯，這就是 Milestone 7 反查高亮的基礎。


In [ ]:
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

cur.execute("SELECT name, type FROM entities")
nodes = [{"id": n, "name": n, "type": t, "val": 1} for n, t in cur.fetchall()]

cur.execute("SELECT source_name, target_name, relation_type FROM relations")
links = [{"source": s, "target": t, "name": r} for s, t, r in cur.fetchall()]

graph_data = {"nodes": nodes, "links": links}

print(f"圖譜共 {len(nodes)} 個節點、{len(links)} 條邊")
print()
print("前 5 個節點：", [n["id"] for n in nodes[:5]])
print("前 5 條邊：")
for l in links[:5]:
    print(f"  ({l['source']}) -[{l['name']}]-> ({l['target']})")

# 反查：找出與某節點相關的 chunk
target_node = nodes[0]["id"] if nodes else None
if target_node:
    cur.execute(
        "SELECT DISTINCT chunk_id FROM relations WHERE source_name = ? OR target_name = ?",
        (target_node, target_node),
    )
    chunk_ids = [row[0] for row in cur.fetchall()]
    print(f"\n節點 {target_node} 出現於 chunks: {chunk_ids}")

conn.close()


## Step 9 — Hybrid 模式：從問題抽 entity 做高亮

當 `mode == "hybrid"` 時，後端會額外讓 LLM 從**使用者問題**抽出關鍵實體名詞，
回傳給前端用來在圖譜上「高亮對應節點」。

這個 cell 示範同樣邏輯。回傳的字串就是要對應到前一步 `nodes[].id` 的名字。


In [ ]:
hybrid_prompt = f'從以下使用者的問題中擷取最關鍵的名詞實體，用 JSON 陣列表示，例如 ["機器學習", "蘋果公司"]。問題：{USER_QUERY}'

res = kg_model.generate_content(hybrid_prompt)
try:
    highlighted = json.loads(res.text)
except Exception as e:
    print("Hybrid 解析失敗：", e)
    print("原始輸出：", res.text)
    highlighted = []

print("應該被高亮的節點：", highlighted)

# 比對哪些真的在我們的 graph 裡
node_ids = {n["id"] for n in nodes}
matched = [h for h in highlighted if h in node_ids]
unmatched = [h for h in highlighted if h not in node_ids]
print(f"圖譜中命中：{matched}")
print(f"圖譜中沒有的：{unmatched}")


## 收工 — 對應到 main 分支的後端

| Notebook 步驟 | 對應後端位置 |
|---|---|
| Step 1 Parsing | `RagEngine.process_document` 前段 |
| Step 2 Chunking | `RagEngine.text_splitter` |
| Step 3 Embedding + Chroma | `RagEngine.collection.add(...)` |
| Step 4 Retrieval | `RagEngine.query` 內 `collection.query(...)` |
| Step 5 LLM 生成 | `RagEngine.query` 內 `gen_model.generate_content(...)` |
| Step 6 KG 抽取 | `RagEngine.extract_and_store_kg` |
| Step 7 SQLite 寫入 | `GraphDB.add_relation` |
| Step 8 圖譜查詢 | `GraphDB.get_graph_data` / `get_chunk_ids_by_node` |
| Step 9 Hybrid 高亮 | `RagEngine.query` 內 `mode == "hybrid"` 分支 |

## 下一步
- 想看 FastAPI 包裝？切到 `main` 分支讀 `backend/app/main.py`。
- 想看 GraphRAG 對照組？切到 `feature/semantic-graphrag` 分支。
- 想清空 ChromaDB 與圖譜重來：刪掉 `./local_chromadb/` 與 `./local_graph.db` 即可。
